In [28]:
import re
import httpx, asyncio
import anthropic
from datetime import date

In [35]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "-", text)
    return text[:80]

In [3]:
slugify('The cat sat on a mat')

'the-cat-sat-on-a-mat'

In [4]:
from fastcore.utils import *

In [5]:
in_jupyter()

True

In [6]:
url = "https://en.wikipedia.org/wiki/Red_panda"

In jupyter there is already an async loop running so we need to use `async`. For CLI where there is no concurrency we can use `sync`.

In [7]:
async def fetch_as_markdown(url: str) -> tuple[str, str]:
    from playwright.async_api import async_playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto(url, wait_until="domcontentloaded", timeout=30000)
        title = await page.title() or "Untitled"
        content = await page.evaluate("""() => {
            const sel = ['article', 'main', '[role="main"]', '.post-content',
                         '.article-content', '.entry-content', 'body'];
            for (const s of sel) {
                const el = document.querySelector(s);
                if (el) return el.innerText;
            }
            return document.body.innerText;
        }""")
        await browser.close()
        return title, content

In [8]:
t, c = await fetch_as_markdown(url)

In [9]:
async def download_images(url: str, vault: Path) -> list[str]:
    from playwright.async_api import async_playwright
    images_dir = vault / "raw" / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto(url, wait_until="domcontentloaded", timeout=30000)
        img_urls = await page.evaluate("""() => {
            return Array.from(document.querySelectorAll('article img, main img'))
                .map(i => i.src).filter(s => s.startsWith('http')).slice(0, 10);
        }""")
        for i, img_url in enumerate(img_urls):
            try:
                ext = img_url.split("?")[0].rsplit(".", 1)[-1][:4] or "jpg"
                fname = images_dir / f"{slugify(url[:40])}-{i}.{ext}"
                resp = await page.request.get(img_url, timeout=10000)
                if resp.ok: fname.write_bytes(await resp.body()); saved.append(str(fname.relative_to(vault)))
                else: await resp.dispose()
            except Exception as e: print(f"Failed {img_url}: {e}")
    return saved

In [10]:
saved = await download_images(url, vault=Path('.'))

This is a review gate — a human-in-the-loop pause before anything gets written to disk.

The flow is:

Claude summarizes the article (takeaways, connections, surprising points)
You read the summary and decide if this article is worth keeping
Press Enter → save proceeds; Ctrl+C → abort, nothing is written
It's essentially a quality filter to avoid cluttering the vault with articles you fetched but then decided weren't useful after seeing the summary.

In [15]:
def interactive_discuss(content: str, title: str, client: anthropic.Anthropic) -> None:
    """Call Claude to discuss key takeaways, then pause before saving."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": (
                f"I just fetched this article: '{title}'\n\n"
                f"{content[:6000]}\n\n"
                "Please discuss:\n"
                "1. The 3-5 most important takeaways\n"
                "2. What concepts this connects to\n"
                "3. Any surprising or counterintuitive points\n"
                "Keep it conversational, ~200 words."
            )
        }]
    )
    print("\n--- Key Takeaways Discussion ---")
    print(response.content[0].text)
    print("--------------------------------")
    input("\nPress Enter to save to vault, or Ctrl+C to abort ... ")


In [19]:
MODEL = "claude-sonnet-4-6"

In [24]:
interactive_discuss(content=c,title=t, client=anthropic.Anthropic())


--- Key Takeaways Discussion ---
# Red Panda Discussion

## Key Takeaways
1. **Not related to giant pandas** - despite the shared name and both using "false thumbs" for bamboo grasping, red pandas are closer to raccoons, weasels, and skunks
2. **Ancient lineage** - their family tree stretches back 18-25 million years, making them a genuinely old evolutionary line
3. **Two subspecies may actually be separate species** - a 2020 genetic study suggests they diverged 250,000 years ago, which is significant enough for species-level distinction
4. **Endangered and under real pressure** - poaching plus habitat fragmentation is a tough combination to survive

## Connections
- **Convergent evolution** is the big one here - red and giant pandas independently evolved the same "false thumb" solution for handling bamboo, which is a classic textbook example
- Connects to **biogeography** and how glaciation events shaped species divergence
- Links to **conservation biology** and captive breeding prog


Press Enter to save to vault, or Ctrl+C to abort ...  


In [31]:
def save_to_vault(url: str, title: str, content: str, tags: list[str], vault: Path, images: list[str]) -> Path:
    slug = slugify(title)
    out_path = vault / "raw" / "web" / f"{slug}.md"

    img_section = ""
    if images: img_section = "\n## Images\n" + "\n".join(f"![[{p}]]" for p in images) + "\n"

    frontmatter = f"""---
title: "{title}"
source: "{url}"
date_ingested: "{date.today()}"
type: web_article
tags: [{", ".join(tags)}]
compiled: false
---

"""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(frontmatter + content + img_section, encoding="utf-8")
    return out_path

In [32]:
save_to_vault(url, t, c, ['pandas', 'wiki', 'other'], Path('.'), saved)

Path('raw/web/red-panda-wikipedia.md')

In [34]:
def update_meta(vault: Path) -> None:
    meta_path = vault / "wiki" / "_meta.md"
    if not meta_path.exists(): return
    text = meta_path.read_text(encoding="utf-8")
    raw_count = len(list((vault / "raw").rglob("*.md")))
    text = re.sub(r"- \*\*Sources ingested\*\*: \d+", f"- **Sources ingested**: {raw_count}", text)
    meta_path.write_text(text, encoding="utf-8")